# Clasificación de sentimientos de reseñas de películas en IMDb con redes neuronales recurrentes

**Miniproyecto 2 · Técnicas de Deep Learning (MAIA-4213) · Maestría en Inteligencia Artificial, Universidad de los Andes**

**Autores:** Miguel Muñoz, David Cardenas y Mario Celis

**Objetivo (enunciado):** desarrollar un método basado en redes neuronales recurrentes que clasifique con precisión las reseñas de películas de IMDb como **positivas** o **negativas**.

**Dataset:** [IMDB Movie Ratings Sentiment Analysis (Kaggle)](https://www.kaggle.com/datasets/yasserh/imdb-movie-ratings-sentiment-analysis/data), versionado en `data/raw/movie.csv`.

| # | Actividad del enunciado | Sección |
|---|---|---|
| 1 | Pipeline de limpieza y tokenización; secuencias numéricas con embeddings (Word2Vec o GloVe) | 2 a 5 |
| 2 | Arquitectura recurrente (LSTM o GRU) con embedding, capa recurrente y capa densa con sigmoide; justificar cada capa | 6 |
| 3 | División train / validación / test; entrenamiento; precision, recall y F1-score en test | 3, 7 a 10 |

**Flujo:** configuración → exploración → limpieza, deduplicación y partición → tokenización y vocabulario → Word2Vec → arquitectura → entrenamiento → línea base → experimentos (LSTM vs GRU, embeddings, semillas) → resultados cuantitativos y cualitativos → resumen para el informe.

Todas las cifras citadas en el informe provienen de las salidas de este notebook.

## 0. Configuración y reproducibilidad

In [ ]:
import os, re, html, json, time, random, copy, platform, collections
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.nn.utils.rnn import pack_padded_sequence

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, roc_curve)
from gensim.models import Word2Vec
import gensim

# Raíz del repositorio (funciona desde notebooks/ o desde la raíz)
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "raw" / "movie.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data" / "raw" / "movie.csv").exists(), "Ejecute el notebook dentro del repositorio."

QUICK = os.environ.get("MP2_QUICK", "0") == "1"   # prueba de humo: subconjunto y 1 época
RETRAIN = False                                   # True: reentrena aunque existan resultados guardados

OUT = ROOT / "outputs" / ("_smoke_test" if QUICK else "")
TAB, FIG, MODELS, RUNS = OUT / "tables", OUT / "figures", OUT / "models", OUT / "tables" / "runs"
SPLITS, REPORT_FIGS = ROOT / "data" / "splits", ROOT / "report" / "figs"
for d in [TAB, FIG, MODELS, RUNS, SPLITS, REPORT_FIGS]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = dict(
    seed=42,
    test_size=0.15, val_size=0.15,       # partición 70 / 15 / 15
    min_freq=2, max_vocab=20000,         # vocabulario construido solo con train
    max_len=200,                         # tokens por reseña que ve la red
    emb_dim=100,                         # dimensión de los embeddings (Word2Vec y aleatorios)
    w2v_window=5, w2v_epochs=5, w2v_sg=1,
    hidden=64, bidirectional=True, dropout=0.3,
    batch_size=128, lr=1e-3, weight_decay=0.0, clip=1.0,
    max_epochs=6, patience=2,
    seeds=[42, 7, 123],
)
if QUICK:
    CONFIG.update(max_epochs=1, seeds=[42, 7])

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
    torch.set_num_threads(max(1, os.cpu_count() or 1))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
print(f"Python {platform.python_version()} | torch {torch.__version__} | gensim {gensim.__version__}")
print("Dispositivo:", DEVICE, "| hilos CPU:", torch.get_num_threads(), "| modo rápido:", QUICK)
print(json.dumps(CONFIG, indent=1))

## 1. Exploración del dataset

Se revisa el balance de clases, la longitud de las reseñas, la presencia de marcas HTML y los duplicados. Etiquetas: `0` = negativa, `1` = positiva.

In [ ]:
raw = pd.read_csv(ROOT / "data/raw/movie.csv", encoding="utf-8-sig")
raw["id"] = np.arange(len(raw))
raw["n_palabras"] = raw.text.str.split().str.len()

print("Reseñas:", len(raw), "| columnas:", list(raw.columns[:2]), "| nulos:", int(raw.isna().sum().sum()))
bal = raw.label.value_counts().sort_index().rename({0: "negativa (0)", 1: "positiva (1)"})
display(bal.to_frame("reseñas").assign(**{"%": (100 * bal / len(raw)).round(1)}))
print("Palabras por reseña:", raw.n_palabras.describe(percentiles=[.5, .75, .9, .95]).round(0).to_dict())
print("Mediana de palabras por clase:", raw.groupby("label").n_palabras.median().to_dict())
print(f"Reseñas con etiquetas HTML <br />: {raw.text.str.contains('<br', regex=False).mean():.1%}")
print("Duplicados exactos (filas sobrantes):", int(raw.duplicated("text").sum()))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for lab, name in [(0, "negativa"), (1, "positiva")]:
    ax[0].hist(raw.loc[raw.label == lab, "n_palabras"].clip(upper=1200), bins=60, alpha=0.55, label=name)
ax[0].axvline(CONFIG["max_len"], color="k", ls="--", lw=1, label=f"max_len = {CONFIG['max_len']}")
ax[0].set_xlabel("Palabras por reseña (recortado en 1200)"); ax[0].set_ylabel("Reseñas"); ax[0].legend()
ax[0].set_title("Longitud de las reseñas por clase")
bal.plot.bar(ax=ax[1], color=["#C44E52", "#4C72B0"]); ax[1].set_title("Balance de clases"); ax[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.savefig(FIG / "eda_longitud_balance.png", dpi=150); plt.show()

**Observaciones**
- Las clases están prácticamente balanceadas, así que *accuracy* y F1 son comparables y no se requieren pesos de clase.
- La longitud varía mucho (cola larga) y es similar entre clases: la longitud por sí sola no delata la etiqueta.
- Más de la mitad de las reseñas contiene marcas HTML `<br />`, que deben eliminarse.
- Hay reseñas duplicadas. Si una copia queda en entrenamiento y otra en prueba, el modelo se evalúa con textos que ya vio (fuga de datos). Se eliminan antes de partir.

## 2. Limpieza, deduplicación y partición

**Limpieza:** se decodifican entidades HTML, se eliminan etiquetas (`<br />`), se pasa a minúsculas y se tokeniza con una expresión regular que conserva:
- **contracciones y negaciones** como un solo token (`didn't`, `isn't`), porque invierten el sentimiento;
- los signos **`!` y `?`**, que suelen intensificar la opinión.

No se eliminan *stopwords*: palabras como *not*, *no* o *but* son decisivas para la polaridad.

**Deduplicación:** dos reseñas son duplicadas si coinciden después de la limpieza. Se conserva una sola copia.

**Partición:** estratificada por clase, 70 % entrenamiento, 15 % validación y 15 % prueba, con semilla fija. Los identificadores se guardan en `data/splits/split.csv`.

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9à-ÿ]+(?:'[a-z]+)?|[!?]")

def clean_text(t: str) -> str:
    t = html.unescape(t)
    t = re.sub(r"<br\s*/?>", " ", t, flags=re.I)
    t = re.sub(r"<[^>]+>", " ", t)
    return t.lower().replace("’", "'").replace("`", "'")

def tokenize(t: str) -> list:
    return TOKEN_RE.findall(clean_text(t))

ejemplo = "This movie was <br /><br />NOT good... I didn't like it at all!!"
print(ejemplo, "\n->", tokenize(ejemplo))

raw["tokens"] = raw.text.map(tokenize)
raw["n_tokens"] = raw.tokens.str.len()
raw["clave"] = raw.tokens.str.join(" ")
dup_mask = raw.duplicated("clave")
conflict = (raw[raw.duplicated("clave", keep=False)].groupby("clave").label.nunique() > 1).sum()
print(f"Duplicados tras la limpieza: {int(dup_mask.sum())} filas sobrantes | grupos con etiquetas distintas: {int(conflict)}")
df = raw[~dup_mask].reset_index(drop=True)
print("Reseñas únicas:", len(df))

SPLIT_PATH = SPLITS / "split.csv"
if SPLIT_PATH.exists():
    split = pd.read_csv(SPLIT_PATH)
    print("Partición cargada desde", SPLIT_PATH.relative_to(ROOT))
else:
    tr_val, te = train_test_split(df.id, test_size=CONFIG["test_size"], stratify=df.label, random_state=CONFIG["seed"])
    tr, va = train_test_split(tr_val, test_size=CONFIG["val_size"] / (1 - CONFIG["test_size"]),
                              stratify=df.set_index("id").loc[tr_val, "label"], random_state=CONFIG["seed"])
    split = pd.concat([pd.DataFrame({"id": tr, "split": "train"}), pd.DataFrame({"id": va, "split": "val"}),
                       pd.DataFrame({"id": te, "split": "test"})]).sort_values("id")
    split.to_csv(SPLIT_PATH, index=False)
    print("Partición creada y guardada en", SPLIT_PATH.relative_to(ROOT))

df = df.merge(split, on="id")
tabla = pd.crosstab(df.split, df.label).loc[["train", "val", "test"]].rename(columns={0: "negativa", 1: "positiva"})
tabla["total"] = tabla.sum(axis=1); tabla["%"] = (100 * tabla.total / tabla.total.sum()).round(1)
display(tabla); tabla.to_csv(TAB / "particion.csv")
assert df.groupby("clave").split.nunique().max() == 1, "Hay textos repetidos entre particiones"
print("Verificación: ningún texto aparece en más de una partición.")

if QUICK:
    df = pd.concat([g.sample(min(len(g), {"train": 2000, "val": 500, "test": 500}[s]), random_state=0)
                    for s, g in df.groupby("split")])
    print("MODO RÁPIDO:", df.split.value_counts().to_dict())

TR, VA, TE = (df[df.split == s].reset_index(drop=True) for s in ["train", "val", "test"])

## 3. Vocabulario y longitud de secuencia

**Vocabulario:** se construye **solo con entrenamiento** (evita fuga de información de validación y prueba), con las palabras de frecuencia ≥ `min_freq` hasta `max_vocab` términos. Índice 0 = `<pad>` (relleno), 1 = `<unk>` (palabra desconocida).

**Longitud:** las RNN procesan los tokens uno a uno, así que el costo crece con la longitud. Se fija `max_len = 200` tokens: cubre completa a la mayoría de reseñas y mantiene el entrenamiento viable en CPU.

**Qué parte conservar de las reseñas largas.** Antes de entrenar las RNN se compara, con un modelo lineal rápido (TF-IDF + regresión logística, sección 7), la F1 en validación al conservar el inicio, el final, o el inicio y el final de cada reseña. La estrategia ganadora se usa en todas las RNN.

In [ ]:
counter = collections.Counter(t for toks in TR.tokens for t in toks)
itos = ["<pad>", "<unk>"] + [w for w, c in counter.most_common(CONFIG["max_vocab"]) if c >= CONFIG["min_freq"]]
stoi = {w: i for i, w in enumerate(itos)}
V = len(itos)
print(f"Tokens distintos en train: {len(counter):,} | vocabulario: {V:,}")
for name, part in [("val", VA), ("test", TE)]:
    toks = [t for ts in part.tokens for t in ts]
    print(f"Tasa de tokens fuera del vocabulario en {name}: {np.mean([t not in stoi for t in toks]):.2%}")
cov = (TR.n_tokens <= CONFIG["max_len"]).mean()
print(f"Reseñas de train con <= {CONFIG['max_len']} tokens (se ven completas): {cov:.1%}")

def truncate(tokens, mode, L=CONFIG["max_len"]):
    if len(tokens) <= L or mode == "completa":
        return tokens
    if mode == "inicio":
        return tokens[:L]
    if mode == "final":
        return tokens[-L:]
    if mode == "inicio+final":
        h = L // 4
        return tokens[:h] + tokens[-(L - h):]
    raise ValueError(mode)

## 4. Embeddings Word2Vec

Se entrena **Word2Vec** [3] (*skip-gram*, dimensión 100, ventana 5) **solo con las reseñas de entrenamiento**. Así los vectores capturan el uso de las palabras en el dominio de reseñas de cine y no se filtra información de validación ni prueba.

Se prefirió Word2Vec entrenado en el dominio en lugar de GloVe [11] preentrenado por dos razones. Primero, es reproducible sin descargar archivos externos de cientos de MB. Segundo, aprende el vocabulario propio de las reseñas (títulos, jerga, faltas de ortografía frecuentes).

La matriz resultante inicializa la capa de embedding de la red. Se compara con embeddings **aleatorios** aprendidos desde cero (sección 8).

In [ ]:
W2V_PATH = MODELS / "w2v_embeddings.npy"
if W2V_PATH.exists() and not RETRAIN and not QUICK:
    EMB_W2V = np.load(W2V_PATH)
    w2v = None
    print("Matriz Word2Vec cargada:", EMB_W2V.shape)
else:
    t0 = time.time()
    w2v = Word2Vec(sentences=TR.tokens.tolist(), vector_size=CONFIG["emb_dim"], window=CONFIG["w2v_window"],
                   min_count=CONFIG["min_freq"], sg=CONFIG["w2v_sg"], epochs=CONFIG["w2v_epochs"],
                   workers=max(1, os.cpu_count() or 1), seed=CONFIG["seed"])
    print(f"Word2Vec entrenado en {time.time() - t0:.0f} s | palabras: {len(w2v.wv):,}")
    rng = np.random.default_rng(CONFIG["seed"])
    EMB_W2V = rng.normal(0, 0.1, (V, CONFIG["emb_dim"])).astype(np.float32)
    EMB_W2V[0] = 0.0                                            # <pad>
    hits = 0
    for w, i in stoi.items():
        if w in w2v.wv:
            EMB_W2V[i] = w2v.wv[w]; hits += 1
    print(f"Cobertura del vocabulario con Word2Vec: {hits / V:.1%}")
    np.save(W2V_PATH, EMB_W2V)

# Verificación cualitativa: vecinos más cercanos (similitud coseno) en el espacio de embeddings
E = EMB_W2V / (np.linalg.norm(EMB_W2V, axis=1, keepdims=True) + 1e-8)
vecinos = {}
for w in ["excellent", "terrible", "boring", "masterpiece", "funny", "waste"]:
    if w in stoi:
        sims = E @ E[stoi[w]]
        vecinos[w] = ", ".join(itos[j] for j in np.argsort(-sims)[1:7])
display(pd.Series(vecinos, name="vecinos más cercanos").to_frame())

## 5. Pipeline de datos: secuencias numéricas y lotes

Cada reseña se convierte en una secuencia de índices del vocabulario (truncada a `max_len`) y su longitud real. Para ahorrar cómputo, los lotes se arman con **reseñas de longitud parecida** (*bucketing*): se rellena poco con `<pad>` y la RNN, gracias a `pack_padded_sequence`, no procesa el relleno.

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, frame, mode):
        self.seqs = [[stoi.get(t, 1) for t in truncate(toks, mode)] or [1] for toks in frame.tokens]
        self.labels = frame.label.values.astype(np.float32)
        self.ids = frame.id.values
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, i):
        return self.seqs[i], self.labels[i]

class BucketBatchSampler(Sampler):
    # Agrupa índices de longitud similar; baraja el orden de los lotes en cada época
    def __init__(self, lengths, batch_size, shuffle, seed):
        self.lengths, self.bs, self.shuffle, self.seed, self.epoch = np.asarray(lengths), batch_size, shuffle, seed, 0
    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch); self.epoch += 1
        idx = rng.permutation(len(self.lengths)) if self.shuffle else np.arange(len(self.lengths))
        chunk = self.bs * 50
        batches = []
        for s in range(0, len(idx), chunk):
            c = idx[s:s + chunk]
            c = c[np.argsort(-self.lengths[c], kind="stable")]
            batches += [c[j:j + self.bs].tolist() for j in range(0, len(c), self.bs)]
        if self.shuffle:
            rng.shuffle(batches)
        return iter(batches)
    def __len__(self):
        return int(np.ceil(len(self.lengths) / self.bs))

def collate(batch):
    seqs, ys = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    x = torch.zeros(len(seqs), int(lengths.max()), dtype=torch.long)
    for i, s in enumerate(seqs):
        x[i, :len(s)] = torch.tensor(s)
    return x, lengths, torch.tensor(ys)

def make_loader(frame, mode, shuffle, seed=42):
    ds = ReviewDataset(frame, mode)
    sampler = BucketBatchSampler([len(s) for s in ds.seqs], CONFIG["batch_size"], shuffle, seed)
    return DataLoader(ds, batch_sampler=sampler, collate_fn=collate, num_workers=0)

x, l, y = next(iter(make_loader(TR, "inicio", shuffle=True)))
print("Lote de ejemplo:", tuple(x.shape), "| longitudes", l[:6].tolist(), "| etiquetas", y[:6].tolist())
print("Reseña -> índices:", x[0, :12].tolist(), "->", [itos[i] for i in x[0, :12].tolist()])

## 6. Arquitectura

```
Índices (B × T)
 └─ Embedding V × 100 (Word2Vec o aleatorio, entrenable, padding_idx = 0) → Dropout 0,3
 └─ LSTM o GRU bidireccional, 1 capa, 64 unidades por dirección (secuencias empaquetadas)
 └─ Estados ocultos finales [→ ; ←] (128) → Dropout 0,3
 └─ Densa 128 → 1 → sigmoide → P(positiva)
```

**Justificación de cada capa**
- **Embedding:** transforma índices discretos en vectores densos donde palabras de uso similar quedan cerca (sección 4). Se inicializa con Word2Vec [3] para partir de conocimiento semántico y se deja entrenable para adaptarlo a la tarea. `padding_idx` hace que el relleno no aporte gradiente.
- **Dropout sobre embeddings y antes de la salida** [8]: regulariza un modelo cuyo mayor número de parámetros está en el embedding (~2 M), lo que reduce el sobreajuste.
- **Capa recurrente LSTM [1] o GRU [2]:** lee la reseña en orden y mantiene un estado que resume el contexto. Las compuertas mitigan el desvanecimiento del gradiente de las RNN simples [10], necesario en secuencias de cientos de tokens. GRU tiene menos parámetros (2 compuertas frente a 3), por lo que se comparan ambas.
- **Bidireccional** [5]: el estado final hacia adelante resume bien el final de la reseña, pero olvida parte del inicio. La dirección inversa cubre ese caso, y el sentimiento puede expresarse en cualquier parte del texto.
- **Una capa, 64 unidades por dirección:** capacidad suficiente para una tarea binaria y costo viable en CPU. Una red más profunda aumentaría el tiempo y el riesgo de sobreajuste con ~28 000 reseñas.
- **Densa con sigmoide:** produce la probabilidad de la clase positiva, tal como sugiere el enunciado. En entrenamiento la sigmoide va dentro de `BCEWithLogitsLoss` (numéricamente más estable) y en inferencia se aplica explícitamente con umbral 0,5.

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, cell="lstm", bidirectional=True, dropout=0.3, emb_weights=None):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if emb_weights is not None:
            with torch.no_grad():
                self.emb.weight.copy_(torch.as_tensor(emb_weights))
        self.emb_drop = nn.Dropout(dropout)
        rnn_cls = nn.LSTM if cell == "lstm" else nn.GRU
        self.rnn = rnn_cls(emb_dim, hidden, num_layers=1, batch_first=True, bidirectional=bidirectional)
        self.bidirectional = bidirectional
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * (2 if bidirectional else 1), 1)

    def forward(self, x, lengths):
        e = self.emb_drop(self.emb(x))
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)
        h = h[0] if isinstance(h, tuple) else h            # LSTM devuelve (h, c)
        h = torch.cat([h[-2], h[-1]], dim=1) if self.bidirectional else h[-1]
        return self.fc(self.drop(h)).squeeze(1)            # logit; sigmoide en la pérdida / inferencia

for cell in ["lstm", "gru"]:
    m = SentimentRNN(V, CONFIG["emb_dim"], CONFIG["hidden"], cell)
    n_emb = m.emb.weight.numel(); n_tot = sum(p.numel() for p in m.parameters())
    print(f"{cell.upper()}: {n_tot:,} parámetros ({n_emb:,} en el embedding, {n_tot - n_emb:,} en la RNN y la salida)")

## 7. Entrenamiento, evaluación y línea base

**Entrenamiento:** entropía cruzada binaria con logits, Adam [9] (lr 10⁻³, lote 128), *gradient clipping* con norma 1 (evita la explosión del gradiente en RNN [7]), máximo `max_epochs` épocas y *early stopping* con paciencia 2 sobre la **F1 de validación**. Se conserva la época con mejor F1 de validación.

**Métricas** (clase positiva = 1, umbral 0,5): **precision, recall y F1** (las que pide el enunciado), *accuracy*, F1-macro, ROC-AUC y matriz de confusión. El conjunto de prueba se usa solo para el reporte final.

**Línea base:** TF-IDF de unigramas y bigramas con regresión logística. Es un modelo clásico y fuerte para esta tarea; sirve para saber si la RNN aporta frente a un modelo que no considera el orden de las palabras.

In [ ]:
def compute_metrics(y, prob, thr=0.5):
    y = np.asarray(y).astype(int); pred = (np.asarray(prob) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(accuracy=accuracy_score(y, pred), precision=precision_score(y, pred, zero_division=0),
                recall=recall_score(y, pred, zero_division=0), f1=f1_score(y, pred, zero_division=0),
                f1_macro=f1_score(y, pred, average="macro", zero_division=0), roc_auc=roc_auc_score(y, prob),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

@torch.no_grad()
def predict(model, loader):
    model.eval(); probs, ys, ids_order = [], [], []
    for x, lengths, y in loader:
        probs.append(torch.sigmoid(model(x.to(DEVICE), lengths)).cpu()); ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(probs).numpy()

def ordered_ids(loader):
    # ids en el mismo orden en que el loader (sin barajar) entrega los ejemplos
    return np.concatenate([loader.dataset.ids[b] for b in loader.batch_sampler])

def run_experiment(tag, cell, emb, seed, mode):
    res_path, hist_path, pred_path = RUNS / f"{tag}.json", RUNS / f"{tag}_historia.csv", RUNS / f"{tag}_pred_test.csv"
    if res_path.exists() and not RETRAIN:
        res = json.loads(res_path.read_text())
        hist = pd.read_csv(hist_path)
        print(f"[{tag}] resultados cargados desde {res_path.relative_to(ROOT)} (entrenado en {res['tiempo_min']:.1f} min, "
              f"{res['dispositivo']})")
        display(hist.round(4))
        return res, hist, pd.read_csv(pred_path)

    set_seed(seed)
    tr_dl, va_dl, te_dl = make_loader(TR, mode, True, seed), make_loader(VA, mode, False), make_loader(TE, mode, False)
    model = SentimentRNN(V, CONFIG["emb_dim"], CONFIG["hidden"], cell, CONFIG["bidirectional"], CONFIG["dropout"],
                         EMB_W2V if emb == "word2vec" else None).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    ckpt_path = MODELS / f"{tag}_ckpt.pt"
    hist, best, best_state, wait, start_ep, elapsed = [], -1.0, None, 0, 1, 0.0
    if ckpt_path.exists() and not RETRAIN:                 # reanuda un entrenamiento interrumpido
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); hist = ck["hist"]
        best, best_state, wait, start_ep, elapsed = ck["best"], ck["best_state"], ck["wait"], ck["epoch"] + 1, ck["elapsed"]
        tr_dl.batch_sampler.epoch = ck["epoch"]
        print(f"[{tag}] reanudando desde la época {start_ep}")
    for ep in range(start_ep, CONFIG["max_epochs"] + 1):
        if wait >= CONFIG["patience"]:
            break
        t0 = time.time(); model.train(); tl, tc, tn = 0.0, 0, 0
        for x, lengths, y in tr_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x, lengths)
            loss = F.binary_cross_entropy_with_logits(logits, y)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG["clip"]); opt.step()
            tl += loss.item() * len(y); tc += ((logits > 0).float() == y).sum().item(); tn += len(y)
        yv, pv = predict(model, va_dl); mv = compute_metrics(yv, pv)
        vloss = float(F.binary_cross_entropy(torch.tensor(pv).clamp(1e-7, 1 - 1e-7), torch.tensor(yv)))
        elapsed += time.time() - t0
        hist.append(dict(epoch=ep, train_loss=tl / tn, train_acc=tc / tn, val_loss=vloss, val_acc=mv["accuracy"],
                         val_f1=mv["f1"], seg=time.time() - t0))
        mejora = mv["f1"] > best
        if mejora:
            best, best_state, wait = mv["f1"], copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
        print(f"[{tag}] ép {ep} | train loss {tl/tn:.4f} acc {tc/tn:.4f} | val loss {vloss:.4f} acc {mv['accuracy']:.4f} "
              f"F1 {mv['f1']:.4f} | {time.time() - t0:.0f}s{' *' if mejora else ''}")
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), hist=hist, best=best, best_state=best_state,
                        wait=wait, epoch=ep, elapsed=elapsed), ckpt_path)
    hist = pd.DataFrame(hist)
    model.load_state_dict(best_state)
    torch.save(best_state, MODELS / f"{tag}.pt")
    yt, pt = predict(model, te_dl)
    pred = pd.DataFrame({"id": ordered_ids(te_dl), "y": yt.astype(int), "prob": pt})
    best_row = hist.loc[hist.val_f1.idxmax()]
    res = dict(tag=tag, cell=cell, emb=emb, seed=seed, truncado=mode, mejor_epoca=int(best_row.epoch),
               epocas=int(len(hist)), train_acc=float(best_row.train_acc), val_acc=float(best_row.val_acc),
               val_f1=float(best_row.val_f1), tiempo_min=elapsed / 60, dispositivo=str(DEVICE),
               parametros=int(sum(p.numel() for p in model.parameters())),
               **{f"test_{k}": float(v) for k, v in compute_metrics(yt, pt).items()})
    hist.to_csv(hist_path, index=False); pred.to_csv(pred_path, index=False)
    res_path.write_text(json.dumps(res, indent=1))
    ckpt_path.unlink(missing_ok=True)
    return res, hist, pred

In [ ]:
# 7.1 Línea base TF-IDF + regresión logística y elección del truncamiento
def tfidf_lr(train_docs, y_train, eval_docs, C):
    vec = TfidfVectorizer(analyzer=lambda d: d, min_df=2, max_features=200_000, sublinear_tf=True)
    # analyzer recibe la lista de tokens; se agregan bigramas manualmente
    add_bi = lambda toks: toks + [a + " " + b for a, b in zip(toks, toks[1:])]
    Xtr = vec.fit_transform([add_bi(t) for t in train_docs]); clf = LogisticRegression(C=C, max_iter=2000)
    clf.fit(Xtr, y_train)
    return [clf.predict_proba(vec.transform([add_bi(t) for t in docs]))[:, 1] for docs in eval_docs]

t0 = time.time()
filas = []
for mode in ["completa", "inicio", "final", "inicio+final"]:
    trd = [truncate(t, mode) for t in TR.tokens]; vad = [truncate(t, mode) for t in VA.tokens]
    (pv,) = tfidf_lr(trd, TR.label, [vad], C=4.0)
    filas.append(dict(texto_usado=mode, val_f1=f1_score(VA.label, pv >= 0.5), val_acc=accuracy_score(VA.label, pv >= 0.5)))
trunc_tab = pd.DataFrame(filas).set_index("texto_usado")
display(trunc_tab.round(4)); trunc_tab.to_csv(TAB / "eleccion_truncamiento.csv")
MODE = trunc_tab.drop(index="completa").val_f1.idxmax()
print(f"Truncamiento elegido para las RNN (mejor F1 en validación con {CONFIG['max_len']} tokens): '{MODE}'")

BASE_PATH = TAB / "linea_base_tfidf.json"
if BASE_PATH.exists() and not RETRAIN and not QUICK:
    base = json.loads(BASE_PATH.read_text()); pred_base = pd.read_csv(TAB / "linea_base_pred_test.csv")
    print("Línea base cargada")
else:
    grid = []
    for C in [0.5, 1.0, 2.0, 4.0, 8.0]:
        (pv,) = tfidf_lr(TR.tokens, TR.label, [VA.tokens], C)
        grid.append((f1_score(VA.label, pv >= 0.5), C))
    best_f1, best_C = max(grid)
    pv, pt = tfidf_lr(TR.tokens, TR.label, [VA.tokens, TE.tokens], best_C)
    base = dict(modelo="TF-IDF (1-2 gramas) + regresión logística", C=best_C, val_f1=f1_score(VA.label, pv >= 0.5),
                **{f"test_{k}": float(v) for k, v in compute_metrics(TE.label, pt).items()})
    pred_base = pd.DataFrame({"id": TE.id, "y": TE.label, "prob": pt})
    BASE_PATH.write_text(json.dumps(base, indent=1)); pred_base.to_csv(TAB / "linea_base_pred_test.csv", index=False)
print(f"Línea base (C = {base['C']}): val F1 {base['val_f1']:.4f} | test F1 {base['test_f1']:.4f} | "
      f"test acc {base['test_accuracy']:.4f} | {time.time() - t0:.0f} s")

## 8. Experimento 1: LSTM vs GRU y embeddings Word2Vec vs aleatorios

Diseño factorial 2 × 2 con la misma partición, semilla (42), hiperparámetros y truncamiento: celda recurrente {LSTM, GRU} × inicialización del embedding {Word2Vec, aleatoria}. La mejor configuración se elige por **F1 en validación** (no con prueba).

Cada corrida guarda en `outputs/tables/runs/` su historia, sus métricas y sus predicciones en prueba. Si ya existen, se cargan en lugar de reentrenar (en CPU cada corrida toma del orden de media hora). Si el entrenamiento se interrumpe, se reanuda desde la última época guardada.

In [ ]:
RESULTS, HIST, PREDS = [], {}, {}
for cell in ["lstm", "gru"]:
    for emb in ["word2vec", "aleatorio"]:
        tag = f"{cell}_{emb}_s{CONFIG['seed']}"
        res, HIST[tag], PREDS[tag] = run_experiment(tag, cell, emb, CONFIG["seed"], MODE)
        RESULTS.append(res)

e1 = pd.DataFrame(RESULTS)
cols = ["cell", "emb", "parametros", "mejor_epoca", "train_acc", "val_acc", "val_f1", "test_precision", "test_recall",
        "test_f1", "test_accuracy", "tiempo_min"]
e1_tab = e1[cols].rename(columns={"cell": "celda", "emb": "embedding"})
e1_tab["celda"] = e1_tab.celda.str.upper()
display(e1_tab.round(4)); e1_tab.to_csv(TAB / "exp1_lstm_gru_embeddings.csv", index=False)
best = e1.loc[e1.val_f1.idxmax()]
BEST_CELL, BEST_EMB = best.cell, best.emb
print(f"Mejor configuración por F1 de validación: {BEST_CELL.upper()} + embeddings {BEST_EMB} (val F1 = {best.val_f1:.4f})")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for tag, h in HIST.items():
    name = tag.rsplit("_s", 1)[0].replace("_", " + ").upper().replace("WORD2VEC", "W2V").replace("ALEATORIO", "aleat.")
    c = ax[0].plot(h.epoch, h.val_loss, marker="o", label=name)[0].get_color()
    ax[0].plot(h.epoch, h.train_loss, ls="--", color=c, alpha=0.6)
    ax[1].plot(h.epoch, h.val_f1, marker="o", color=c, label=name)
    ax[2].plot(h.epoch, h.train_acc - h.val_acc, marker="o", color=c, label=name)
ax[0].set_title("Pérdida (punteada: train)"); ax[1].set_title("F1 en validación"); ax[2].set_title("Brecha accuracy train - val")
for a in ax:
    a.set_xlabel("Época"); a.legend(fontsize=7); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG / "exp1_curvas.png", dpi=150); plt.show()

## 9. Experimento 2: variabilidad entre semillas

La mejor configuración se reentrena con otras dos semillas (cambian la inicialización, el orden de los lotes y el *dropout*). Se reporta **media ± desviación estándar** en prueba para saber qué diferencias del experimento 1 son mayores que el ruido de entrenamiento.

In [ ]:
for seed in CONFIG["seeds"]:
    tag = f"{BEST_CELL}_{BEST_EMB}_s{seed}"
    if tag not in HIST:
        res, HIST[tag], PREDS[tag] = run_experiment(tag, BEST_CELL, BEST_EMB, seed, MODE)
        RESULTS.append(res)

R = pd.DataFrame(RESULTS)
e2 = R[(R.cell == BEST_CELL) & (R.emb == BEST_EMB)].sort_values("seed")
metric_cols = ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]
display(e2[["seed", "mejor_epoca", "val_f1"] + metric_cols].round(4))
agg = e2[metric_cols].agg(["mean", "std"]).T
agg["media ± std"] = agg["mean"].map("{:.4f}".format) + " ± " + agg["std"].map("{:.4f}".format)
display(agg[["media ± std"]]); agg.to_csv(TAB / "exp2_semillas.csv")

## 10. Resultados cuantitativos en prueba

Tabla resumen de todos los modelos, matriz de confusión y curva ROC del mejor modelo (semilla 42), e intervalo de confianza al 95 % de la F1 por *bootstrap* (1 000 remuestreos del conjunto de prueba).

In [ ]:
rows = [dict(modelo=base["modelo"], **{k.replace("test_", ""): base[k] for k in metric_cols})]
for r in RESULTS:
    if r["seed"] == CONFIG["seed"]:
        rows.append(dict(modelo=f"Bi{r['cell'].upper()} + emb. {r['emb']} (semilla 42)", **{k.replace("test_", ""): r[k] for k in metric_cols}))
rows.append(dict(modelo=f"Bi{BEST_CELL.upper()} + emb. {BEST_EMB}, media de {len(e2)} semillas",
                 **{k.replace("test_", ""): e2[k].mean() for k in metric_cols}))
resumen = pd.DataFrame(rows).set_index("modelo")
std_row = pd.Series({k.replace("test_", ""): e2[k].std() for k in metric_cols}, name="  desviación estándar")
resumen = pd.concat([resumen, std_row.to_frame().T])
display(resumen.round(4)); resumen.to_csv(TAB / "resumen_test.csv")

BEST_TAG = f"{BEST_CELL}_{BEST_EMB}_s{CONFIG['seed']}"
P = PREDS[BEST_TAG].merge(TE[["id", "text", "n_tokens"]], on="id")
print(f"\nReporte de clasificación, {BEST_TAG} (prueba):")
print(classification_report(P.y, (P.prob >= 0.5).astype(int), target_names=["negativa", "positiva"], digits=4))

rng = np.random.default_rng(0); y_arr, p_arr = P.y.values, (P.prob.values >= 0.5).astype(int)
boot = [f1_score(y_arr[i], p_arr[i]) for i in (rng.integers(0, len(P), len(P)) for _ in range(1000))]
F1_CI = np.percentile(boot, [2.5, 97.5])
print(f"F1 en prueba: {f1_score(y_arr, p_arr):.4f} (IC 95 % bootstrap: {F1_CI[0]:.4f} - {F1_CI[1]:.4f})")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
cm = confusion_matrix(P.y, (P.prob >= 0.5).astype(int))
ax[0].imshow(cm / cm.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, f"{cm[i, j]}\n({cm[i, j] / cm[i].sum():.1%})", ha="center", va="center",
                   color="white" if cm[i, j] / cm[i].sum() > 0.5 else "black")
ax[0].set_xticks([0, 1], ["negativa", "positiva"]); ax[0].set_yticks([0, 1], ["negativa", "positiva"])
ax[0].set_xlabel("Predicción"); ax[0].set_ylabel("Etiqueta real"); ax[0].set_title(f"Matriz de confusión\n{BEST_TAG}")

for name, pr in [(f"Bi{BEST_CELL.upper()} ({BEST_EMB})", P[["y", "prob"]]), ("TF-IDF + reg. logística", pred_base[["y", "prob"]])]:
    fpr, tpr, _ = roc_curve(pr.y, pr.prob)
    ax[1].plot(fpr, tpr, label=f"{name}: AUC {roc_auc_score(pr.y, pr.prob):.3f}")
ax[1].plot([0, 1], [0, 1], "k--", lw=0.8); ax[1].set_xlabel("Tasa de falsos positivos"); ax[1].set_ylabel("Tasa de verdaderos positivos")
ax[1].set_title("Curva ROC (prueba)"); ax[1].legend(fontsize=8)

# Desempeño por longitud de la reseña: ¿afecta el truncamiento a la RNN?
bins = [0, 100, 200, 400, 10_000]; labels = ["≤100", "101-200", "201-400", ">400"]
B = P.merge(pred_base[["id", "prob"]].rename(columns={"prob": "prob_base"}), on="id")
B["tramo"] = pd.cut(B.n_tokens, bins=bins, labels=labels)
por_long = B.groupby("tramo", observed=True).apply(lambda g: pd.Series({
    "n": len(g), "acc_rnn": accuracy_score(g.y, g.prob >= 0.5), "acc_base": accuracy_score(g.y, g.prob_base >= 0.5)}))
display(por_long.round(4)); por_long.to_csv(TAB / "desempeno_por_longitud.csv")
xx = np.arange(len(por_long))
ax[2].bar(xx - 0.2, por_long.acc_rnn, 0.4, label=f"Bi{BEST_CELL.upper()}"); ax[2].bar(xx + 0.2, por_long.acc_base, 0.4, label="TF-IDF + RL")
ax[2].set_xticks(xx, [f"{t}\n(n={int(n)})" for t, n in zip(por_long.index, por_long.n)])
ax[2].set_ylim(0.75, 1.0); ax[2].set_xlabel("Tokens por reseña"); ax[2].set_ylabel("Accuracy en prueba")
ax[2].axvline(1.5, color="k", ls=":", lw=1); ax[2].set_title(f"Accuracy por longitud (RNN ve {CONFIG['max_len']} tokens)")
ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "resultados_cuantitativos.png", dpi=150); plt.show()

## 11. Resultados cualitativos

Ejemplos de prueba con la reseña, la etiqueta real, la predicción y la probabilidad de clase positiva: aciertos con alta confianza de cada clase y los errores con mayor confianza (falsos positivos y falsos negativos). Se muestra el fragmento que la red efectivamente leyó.

In [ ]:
P["pred"] = (P.prob >= 0.5).astype(int)
P["confianza"] = np.where(P.pred == 1, P.prob, 1 - P.prob)
P["visto"] = P.text.map(lambda t: " ".join(truncate(tokenize(t), MODE)))
nombre = {0: "negativa", 1: "positiva"}
sel = pd.concat([
    P[(P.y == 1) & (P.pred == 1)].nlargest(2, "confianza").assign(caso="acierto"),
    P[(P.y == 0) & (P.pred == 0)].nlargest(2, "confianza").assign(caso="acierto"),
    P[(P.y == 0) & (P.pred == 1)].nlargest(3, "confianza").assign(caso="falso positivo"),
    P[(P.y == 1) & (P.pred == 0)].nlargest(3, "confianza").assign(caso="falso negativo"),
])
cual = pd.DataFrame({"caso": sel.caso, "real": sel.y.map(nombre), "predicha": sel.pred.map(nombre),
                     "P(positiva)": sel.prob.round(3), "tokens": sel.n_tokens,
                     "fragmento leído por la red": sel.visto.str.slice(0, 420) + " ..."})
cual.to_csv(TAB / "ejemplos_cualitativos.csv", index=False)
with pd.option_context("display.max_colwidth", None):
    display(cual)

err = P[P.pred != P.y]
print(f"Errores: {len(err)} de {len(P)} | falsos positivos {int(((P.y == 0) & (P.pred == 1)).sum())} | "
      f"falsos negativos {int(((P.y == 1) & (P.pred == 0)).sum())}")
print(f"Errores con confianza > 0,9: {int((err.confianza > 0.9).sum())} | mediana de tokens en errores: "
      f"{err.n_tokens.median():.0f} vs {P[P.pred == P.y].n_tokens.median():.0f} en aciertos")

## 12. Figuras y cifras para el informe

Se exportan las figuras del informe a `report/figs/` y las cifras citadas a `outputs/tables/resumen_informe.json`, para que el informe y el notebook coincidan.

In [ ]:
plt.rcParams.update({"font.size": 8})
# Figura A: curvas de validación del experimento 1 + matriz de confusión
fig, ax = plt.subplots(1, 2, figsize=(7.0, 2.5), gridspec_kw={"width_ratios": [1.5, 1]})
for tag in [t for t in HIST if t.endswith(f"_s{CONFIG['seed']}")]:
    h = HIST[tag]
    name = tag.rsplit("_s", 1)[0].replace("_", " + ").upper().replace("WORD2VEC", "W2V").replace("ALEATORIO", "aleat.")
    ax[0].plot(h.epoch, h.val_f1, marker="o", ms=3, label=name)
ax[0].set_xlabel("Época"); ax[0].set_ylabel("F1 validación"); ax[0].set_title("(a) Experimento 1: F1 en validación")
ax[0].legend(fontsize=6.5); ax[0].grid(alpha=0.3)
ax[1].imshow(cm / cm.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax[1].text(j, i, f"{cm[i, j]}\n({cm[i, j] / cm[i].sum():.1%})", ha="center", va="center", fontsize=7,
                   color="white" if cm[i, j] / cm[i].sum() > 0.5 else "black")
ax[1].set_xticks([0, 1], ["neg.", "pos."]); ax[1].set_yticks([0, 1], ["neg.", "pos."])
ax[1].set_xlabel("Predicción"); ax[1].set_ylabel("Real"); ax[1].set_title("(b) Matriz de confusión (prueba)")
plt.tight_layout(); plt.savefig(REPORT_FIGS / "fig_curvas_confusion.pdf", bbox_inches="tight"); plt.show()

# Figura B: accuracy por longitud
fig, ax = plt.subplots(figsize=(3.4, 2.3))
ax.bar(xx - 0.2, por_long.acc_rnn, 0.4, label=f"Bi{BEST_CELL.upper()}"); ax.bar(xx + 0.2, por_long.acc_base, 0.4, label="TF-IDF + RL")
ax.set_xticks(xx, [f"{t}\n(n={int(n)})" for t, n in zip(por_long.index, por_long.n)], fontsize=7)
ax.set_ylim(0.75, 1.0); ax.set_xlabel("Tokens por reseña"); ax.set_ylabel("Accuracy (prueba)"); ax.legend(fontsize=7)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(REPORT_FIGS / "fig_longitud.pdf", bbox_inches="tight"); plt.show()
plt.rcParams.update({"font.size": 10})

resumen_informe = {
    "dataset": {"reseñas": int(len(raw)), "duplicados_eliminados": int(dup_mask.sum()), "únicas": int(len(raw) - dup_mask.sum()),
                "partición": tabla.total.to_dict(), "vocabulario": V, "max_len": CONFIG["max_len"], "truncamiento": MODE,
                "cobertura_max_len_train": round(float(cov), 4)},
    "eleccion_truncamiento_val_f1": trunc_tab.val_f1.round(4).to_dict(),
    "linea_base": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in base.items()},
    "experimento_1": e1_tab.round(4).to_dict(orient="records"),
    "mejor_configuracion": f"Bi{BEST_CELL.upper()} + embeddings {BEST_EMB}",
    "experimento_2_media_std": agg["media ± std"].to_dict(),
    "f1_ic95_semilla42": [round(float(v), 4) for v in F1_CI],
    "accuracy_por_longitud": por_long.round(4).to_dict(orient="index"),
    "errores_semilla42": {"fp": int(cm[0, 1]), "fn": int(cm[1, 0])},
    "dispositivo": str(DEVICE), "modo_rapido": QUICK,
}
(TAB / "resumen_informe.json").write_text(json.dumps(resumen_informe, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
print(json.dumps(resumen_informe, indent=1, ensure_ascii=False, default=str))

## 13. Resultados y discusión

*Esta sección se completa con las salidas de la ejecución completa.*

## 14. Referencias

[1] S. Hochreiter and J. Schmidhuber, "Long short-term memory," *Neural Comput.*, vol. 9, no. 8, pp. 1735–1780, 1997.

[2] K. Cho *et al.*, "Learning phrase representations using RNN encoder-decoder for statistical machine translation," in *Proc. Conf. Empirical Methods Natural Lang. Process. (EMNLP)*, 2014, pp. 1724–1734.

[3] T. Mikolov, K. Chen, G. Corrado, and J. Dean, "Efficient estimation of word representations in vector space," arXiv:1301.3781, 2013.

[4] A. L. Maas, R. E. Daly, P. T. Pham, D. Huang, A. Y. Ng, and C. Potts, "Learning word vectors for sentiment analysis," in *Proc. 49th Annu. Meeting Assoc. Comput. Linguistics (ACL-HLT)*, 2011, pp. 142–150.

[5] M. Schuster and K. K. Paliwal, "Bidirectional recurrent neural networks," *IEEE Trans. Signal Process.*, vol. 45, no. 11, pp. 2673–2681, 1997.

[6] J. Chung, C. Gulcehre, K. Cho, and Y. Bengio, "Empirical evaluation of gated recurrent neural networks on sequence modeling," arXiv:1412.3555, 2014.

[7] R. Pascanu, T. Mikolov, and Y. Bengio, "On the difficulty of training recurrent neural networks," in *Proc. Int. Conf. Mach. Learn. (ICML)*, 2013, pp. 1310–1318.

[8] N. Srivastava, G. Hinton, A. Krizhevsky, I. Sutskever, and R. Salakhutdinov, "Dropout: A simple way to prevent neural networks from overfitting," *J. Mach. Learn. Res.*, vol. 15, pp. 1929–1958, 2014.

[9] D. P. Kingma and J. Ba, "Adam: A method for stochastic optimization," in *Proc. Int. Conf. Learn. Represent. (ICLR)*, 2015, arXiv:1412.6980.

[10] Y. Bengio, P. Simard, and P. Frasconi, "Learning long-term dependencies with gradient descent is difficult," *IEEE Trans. Neural Netw.*, vol. 5, no. 2, pp. 157–166, 1994.

[11] J. Pennington, R. Socher, and C. D. Manning, "GloVe: Global vectors for word representation," in *Proc. Conf. Empirical Methods Natural Lang. Process. (EMNLP)*, 2014, pp. 1532–1543.